# Đề 12 – Tìm kiếm ảnh/đối tượng trong video (Colab/Kaggle)
Baseline CLIP vs DINOv2, full-frame vs region, trên DAVIS 2017 val (annotation CC BY 4.0). Bật GPU T4.

In [ ]:
!git clone <REPO_URL> de12 || true
%cd de12
!pip -q install faiss-cpu gradio

In [ ]:
!wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O davis.zip && unzip -q davis.zip -d data && rm davis.zip
!python scripts/prepare_davis.py --davis data/DAVIS --out data/davis_bench

In [ ]:
for m in ['openai/clip-vit-base-patch32','facebook/dinov2-small','google/siglip2-base-patch16-224']:
    n=m.split('/')[-1]
    !python scripts/build_index.py --videos data/davis_bench/videos --out data/idx_{n} --model {m} --fps 1 --regions grid
    for q in ['object','scene']:
        !python scripts/evaluate.py --index data/idx_{n} --queries data/davis_bench/queries_{q}.json --out results/{n}_{q}.json

In [ ]:
# Tùy chọn (GPU): region bằng OWLv2 objectness
!python scripts/build_index.py --videos data/davis_bench/videos --out data/idx_dinov2_owl --model facebook/dinov2-small --fps 1 --regions owlv2
!python scripts/evaluate.py --index data/idx_dinov2_owl --queries data/davis_bench/queries_object.json

In [ ]:
# SAM 3.1 regions (GPU, cần HF_TOKEN có quyền facebook/sam3.1)
# import os; os.environ['HF_TOKEN']=...  # Kaggle: Secrets
!git clone https://github.com/facebookresearch/sam3.git && pip -q install -e sam3
!python scripts/build_index.py --videos data/davis_bench/videos --out data/idx_dinov2_sam3 --model facebook/dinov2-base --fps 1 --regions sam3 --dedup 4
!python scripts/evaluate.py --index data/idx_dinov2_sam3 --queries data/davis_bench/queries_object.json

In [ ]:
# DINOv3 (gated): cần HF token + chấp nhận license tại huggingface.co/facebook
# from huggingface_hub import login; login()
# !python scripts/build_index.py --videos data/davis_bench/videos --out data/idx_dinov3 --model facebook/dinov3-vits16-pretrain-lvd1689m --regions grid
# (tên checkpoint DINOv3 ở trên lấy từ HF; truy cập hiện bị 403 – chưa được cấp quyền)

In [ ]:
!python app.py --index data/idx_dinov2-small --videos data/davis_bench/videos --share